# Chatbot

This notebook uses the same direct Ollama `/api/chat` approach as `../scripts/chat.py`, but presents it as a Jupyter chat panel. It can run as a local text chatbot and optionally speak replies through `sdk_client.Robot`.


In [1]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Configured for iface='eth0', domain_id=0.


Import notebook UI helpers and the standard-library HTTP client used by the Ollama script.


In [2]:
import json
import time
import urllib.error
import urllib.request

import ipywidgets as widgets
from IPython.display import display

from sdk_client import Robot


DEFAULT_SYSTEM_PROMPT = (
    "You are the voice of a Unitree humanoid robot. Chat naturally with nearby people. "
    "Reply in no more than 25 words. "
    "Do not mention that you are a language model. Do not use markdown or hidden reasoning."
)


def clean_reply(text):
    text = str(text).strip()
    while "<think>" in text and "</think>" in text:
        before, rest = text.split("<think>", 1)
        _hidden, after = rest.split("</think>", 1)
        text = (before + after).strip()
    return " ".join(text.split())


Configure Ollama and build the chat state. These defaults mirror `../scripts/chat.py`; override them with environment variables before running the cell.


In [3]:
OLLAMA_URL = os.environ.get("G1_OLLAMA_URL", "http://127.0.0.1:11434").rstrip("/")
MODEL = os.environ.get("G1_CHAT_MODEL", "qwen3.5:9b")
SYSTEM_PROMPT = os.environ.get("G1_CHAT_SYSTEM", DEFAULT_SYSTEM_PROMPT)
TEMPERATURE = float(os.environ.get("G1_CHAT_TEMPERATURE", "0.4"))
TIMEOUT_S = float(os.environ.get("G1_CHAT_TIMEOUT", "30"))
MAX_HISTORY = int(os.environ.get("G1_CHAT_MAX_HISTORY", "4"))
NUM_PREDICT = int(os.environ.get("G1_CHAT_NUM_PREDICT", "48"))
NUM_CTX = int(os.environ.get("G1_CHAT_NUM_CTX", "1024"))
KEEP_ALIVE = os.environ.get("G1_CHAT_KEEP_ALIVE", "15m")
NUM_THREAD = os.environ.get("G1_CHAT_NUM_THREAD")

messages = [{"role": "system", "content": SYSTEM_PROMPT}]
robot = None
speak_replies = False


def post_ollama_chat(body, timeout=TIMEOUT_S):
    data = json.dumps(body).encode("utf-8")
    request = urllib.request.Request(
        f"{OLLAMA_URL}/api/chat",
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=float(timeout)) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama HTTP {exc.code}: {detail}") from exc


def ask_ollama(user_text):
    history = messages + [{"role": "user", "content": str(user_text)}]
    del history[1:max(1, len(history) - max(1, MAX_HISTORY))]
    options = {
        "temperature": float(TEMPERATURE),
        "num_predict": int(NUM_PREDICT),
        "num_ctx": int(NUM_CTX),
    }
    if NUM_THREAD:
        options["num_thread"] = int(NUM_THREAD)
    body = {
        "model": MODEL,
        "messages": history,
        "stream": False,
        "keep_alive": KEEP_ALIVE,
        "think": False,
        "options": options,
    }
    started = time.time()
    result = post_ollama_chat(body)
    reply = clean_reply(result.get("message", {}).get("content", ""))
    if not reply:
        reply = "I heard you, but I am not sure how to answer that yet."
    history.append({"role": "assistant", "content": reply})
    messages[:] = history
    return reply, time.time() - started


def warm_up_ollama():
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "Answer with one short word."},
            {"role": "user", "content": "Ready?"},
        ],
        "stream": False,
        "keep_alive": KEEP_ALIVE,
        "think": False,
        "options": {"temperature": 0, "num_predict": 2, "num_ctx": int(NUM_CTX)},
    }
    post_ollama_chat(body, timeout=TIMEOUT_S)

print(f"Ollama chat ready: url={OLLAMA_URL} model={MODEL}")


Set ANTHROPIC_API_KEY before sending messages.
provider=anthropic model=claude-sonnet-4-6 base=https://api.anthropic.com/v1


Optional robot speech binding. Set `enable_robot_speech = True` before running this cell if replies should be spoken.


In [4]:
enable_robot_speech = False

if enable_robot_speech:
    robot = Robot(iface=IFACE, domain_id=DOMAIN_ID, safety_boot=False, auto_start_sensors=False)
    speak_replies = True
    print("Robot speech enabled.")
else:
    print("Robot speech disabled. Set enable_robot_speech=True and rerun this cell to speak replies.")


Robot tools disabled. Set use_robot_tools=True and rerun this cell to enable them.


Run the chat panel. Use Warm Up once before the first message if the Ollama model is not already loaded.


In [5]:
prompt = widgets.Textarea(placeholder="Type a message...", layout=widgets.Layout(width="100%", height="90px"))
send = widgets.Button(description="Send", button_style="success")
warmup = widgets.Button(description="Warm Up")
clear = widgets.Button(description="Clear")
speak = widgets.Checkbox(value=speak_replies, description="speak replies")
chat_log = widgets.Textarea(layout=widgets.Layout(width="100%", height="420px"), disabled=True)


def add(line):
    chat_log.value = (chat_log.value + line + "\n")[-12000:]


def on_send(_):
    text = prompt.value.strip()
    if not text:
        return
    prompt.value = ""
    add(f"you> {text}")
    try:
        reply, elapsed = ask_ollama(text)
        add(f"bot> {reply}")
        add(f"[ollama] {elapsed:.1f}s")
        if speak.value and robot is not None:
            robot.say(reply)
    except Exception as exc:
        add(f"error> {exc}")


def on_warmup(_):
    try:
        started = time.time()
        warm_up_ollama()
        add(f"[ollama] warm-up finished in {time.time() - started:.1f}s")
    except Exception as exc:
        add(f"error> warm-up failed: {exc}")


def on_clear(_):
    messages[:] = [{"role": "system", "content": SYSTEM_PROMPT}]
    chat_log.value = ""

send.on_click(on_send)
warmup.on_click(on_warmup)
clear.on_click(on_clear)
display(widgets.VBox([prompt, widgets.HBox([send, warmup, clear, speak]), chat_log]))
